# RegWatch — Démonstration du Pipeline RAG
## Analyse automatique de conformité Solvabilité II

**Auteur :** Thiané — Responsable Data/IA  
**Phases démontrées :** 1 (Référentiel) → 2 (Ingestion) → 3 (Embeddings) → 4 (RAG)  
**Stack :** Python 3.11 | ChromaDB | sentence-transformers | LangChain

---
### Architecture du pipeline
```
Référentiel Solvabilité II (71 exigences atomiques)
        ↓ Embedding (multilingual-e5-large en prod)
   ChromaDB [referentiel_solvabilite2]
        
Rapport SFCR (PDF)
        ↓ PyMuPDF → chunking → embedding
   ChromaDB [sfcr_documents]
        
   Pour chaque exigence :
     embed(texte_verification) → recherche top-K passages
        ↓ LLM (Claude Sonnet / Mistral / RuleBased)
     score_final = 0.4×cosinus + 0.6×LLM
        ↓
     COUVERT (≥0.75) / PARTIEL (≥0.45) / ABSENT (<0.45)
```

## Cellule 0 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import shutil
import uuid
from pathlib import Path
import numpy as np

print('✓ Imports OK')

## Phase 1 — Référentiel Solvabilité II
### Ce qu'on a construit
71 exigences atomiques extraites de :
- Directive 2009/138/CE (Solvabilité II)
- Règlement délégué 2015/35
- Guidelines EIOPA-BoS-15/109

Chaque exigence a un **identifiant normalisé**, un **niveau d'obligation** (SHALL/SHOULD/MAY) et un **texte_verification** qui sert de requête RAG.

In [ ]:
from src.ingestion.referentiel_loader import ReferentielLoader
from src.ingestion.referentiel_solvabilite2 import PILIERS, THEMES
from src.ingestion.models import NiveauObligation

loader = ReferentielLoader()
loader.charger()
print(loader.afficher_stats())

In [ ]:
# Exemple d'exigence atomique — ce que le système compare au SFCR
exigence = loader.get_exigence('SII-P3-REG2015-Art298-§1')
print(f'ID            : {exigence.id_exigence}')
print(f'Article       : {exigence.article}')
print(f'Thème         : {exigence.theme_id} — {THEMES[exigence.theme_id].libelle}')
print(f'Obligation    : {exigence.niveau_obligation.value} (poids={exigence.poids_obligation})')
print(f'Texte original: {exigence.texte_original[:150]}...')
print()
print(f'→ Question posée au LLM:')
print(f'  "{exigence.texte_verification}"')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Visualisation : distribution des exigences par thème et pilier
stats = loader.stats()
themes_ids = list(stats['par_theme'].keys())
nb_exig    = list(stats['par_theme'].values())
couleurs   = ['#2196F3']*3 + ['#4CAF50']*4 + ['#FF9800']*4  # P1/P2/P3

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Référentiel Solvabilité II — Structure et distribution', 
             fontsize=14, fontweight='bold')

# Graphe 1 : Exigences par thème
bars = axes[0].barh(themes_ids, nb_exig, color=couleurs)
axes[0].set_xlabel('Nombre d\'exigences')
axes[0].set_title('Exigences par thème')
for bar, nb in zip(bars, nb_exig):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                str(nb), va='center', fontsize=9)
p1 = mpatches.Patch(color='#2196F3', label='Pilier 1 (Quantitatif)')
p2 = mpatches.Patch(color='#4CAF50', label='Pilier 2 (Gouvernance)')
p3 = mpatches.Patch(color='#FF9800', label='Pilier 3 (Reporting)')
axes[0].legend(handles=[p1, p2, p3], loc='lower right', fontsize=8)

# Graphe 2 : Distribution SHALL/SHOULD/MAY
obligation_data = stats['par_obligation']
labels = ['SHALL\n(obligatoire)', 'SHOULD\n(recommandé)', 'MAY\n(optionnel)']
values = [obligation_data['SHALL'], obligation_data['SHOULD'], obligation_data['MAY']]
colors = ['#f44336', '#FF9800', '#4CAF50']
wedges, texts, autotexts = axes[1].pie(
    values, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=90,
    textprops={'fontsize': 10}
)
axes[1].set_title('Répartition par niveau d\'obligation')

plt.tight_layout()
plt.savefig('../docs/referentiel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphe sauvegardé dans docs/')

## Phase 2 — Ingestion & Chunking
### Stratégie de chunking différenciée

| Document | Taille chunk | Overlap | Stratégie |
|---|---|---|---|
| **Référentiel** | 1 exigence = 1 chunk | 0 | Autonomie sémantique |
| **SFCR** | 450 tokens | 80 tokens | `RecursiveCharacterTextSplitter` |

In [ ]:
from src.ingestion.sfcr_chunker import SFCRChunker
from src.ingestion.text_utils import compter_tokens

chunker = SFCRChunker(chunk_size=450, chunk_overlap=80)

# Simulation du chunking d'une section SFCR réelle
texte_section_B = """
Le système de gouvernance de l'entreprise repose sur une séparation claire
des responsabilités entre l'organe d'administration et la direction effective.
Le conseil d'administration, composé de neuf membres dont trois indépendants,
se réunit au moins quatre fois par an. Il est assisté d'un comité d'audit,
d'un comité des risques et d'un comité des rémunérations.

La direction générale assure la gestion opérationnelle de l'entreprise et
rend compte trimestriellement au conseil d'administration des résultats,
des risques majeurs et des évolutions réglementaires significatives.

Les quatre fonctions clés requises par Solvabilité II sont opérationnelles
et indépendantes des fonctions qu'elles supervisent. La fonction de gestion
des risques coordonne le dispositif global de maîtrise des risques. La fonction
de vérification de la conformité assure le respect des obligations réglementaires.
La fonction d'audit interne évalue l'efficacité du système de contrôle interne.
La fonction actuarielle supervise le calcul des provisions techniques.
"""

chunks = chunker.chunker_referentiel(
    texte=texte_section_B,
    source_id='DEMO',
    article='Section B - Demo',
    id_exigence='SII-P2-DIR2009-Art41-§demo',
)

print(f'Texte original : {compter_tokens(texte_section_B)} tokens')
print(f'Nombre de chunks produits : {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'  Chunk {i+1} : {c.nb_tokens} tokens | section={c.section}')
    print(f'    "{c.texte[:80]}..."')

## Phase 3 — Embeddings & Base vectorielle
### Choix technique : multilingual-e5-large

**En production** (avec accès HuggingFace) :
```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('intfloat/multilingual-e5-large')
# dim=1024, multilingue FR/EN, top-3 MTEB leaderboard
```

**En développement offline** (cet environnement) :
```python
engine = LocalTfidfEmbedder(dimension=256)
# TF-IDF + LSA, même interface API, swap en 1 ligne
```

**Pourquoi les préfixes `query:` / `passage:` ?**  
multilingual-e5-large est entraîné avec ces préfixes. Sans eux, les performances baissent de **15-20%** (Thakur et al., BEIR 2021).

In [ ]:
from src.rag.embedding_engine import LocalTfidfEmbedder, creer_embedding_engine
from src.rag.vector_store import VectorStore
from src.ingestion.ingestion_pipeline import IngestionPipeline

# Charger le référentiel et préparer l'engine
pipeline_ing = IngestionPipeline()
resultat_ref = pipeline_ing.ingerer_referentiel()

textes_fit = [t.replace('query: ', '') for t in resultat_ref.textes_embedding]
engine = LocalTfidfEmbedder(dimension=256)
engine.fit(textes_fit)

# Générer les embeddings
embeddings = engine.embed_documents(resultat_ref.textes_embedding)
print(f'Embeddings référentiel : {embeddings.shape}')
print(f'Normalisation L2 : normes = {np.linalg.norm(embeddings[:3], axis=1).round(4)}')

# ChromaDB
demo_dir = Path('/tmp/demo_notebook')
if demo_dir.exists(): shutil.rmtree(demo_dir)
store = VectorStore(persist_dir=demo_dir)
store.initialiser(engine)
nb = store.indexer_referentiel(
    chunks=resultat_ref.chunks,
    embeddings=embeddings,
    ids_exigences=resultat_ref.ids_exigences
)
print(f'\nChromaDB référentiel : {nb} chunks indexés')
print(store.afficher_stats())

## Phase 4 — Pipeline RAG
### Démonstration avec un SFCR synthétique

En production : charger un vrai PDF SFCR via `IngestionPipeline.ingerer_sfcr()`

In [ ]:
from src.rag.llm_engine import RuleBasedLLM, creer_llm_engine
from src.rag.rag_pipeline import RAGPipeline
from src.ingestion.models import ChunkDocument, StatutConformite

# SFCR synthétique représentatif
sfcr_contenu = [
    ('Le conseil d\'administration se réunit 4 fois par an. Les quatre fonctions '
     'clés Solvabilité II sont opérationnelles : gestion des risques, conformité, '
     'audit interne et fonction actuarielle, toutes indépendantes.',
     'B_GOUVERNANCE'),
    ('Le système de contrôle interne repose sur des procédures administratives '
     'et comptables documentées. Cadre de contrôle à 3 niveaux de défense. '
     'Reporting mensuel à la direction générale et trimestriel au CA.',
     'B_GOUVERNANCE'),
    ('La politique de rémunération distingue composante fixe et composante variable '
     'pour les dirigeants effectifs et détenteurs de fonctions clés, '
     'conformément aux exigences Solvabilité II articles 275-276.',
     'B_GOUVERNANCE'),
    ('L\'ORSA 2023 a évalué les besoins globaux de solvabilité en tenant compte '
     'du profil de risque spécifique. Le SCR reste couvert dans tous les scénarios '
     'adverses testés (choc taux, choc marchés, choc mortalité).',
     'C_PROFIL_RISQUE'),
    ('SCR formule standard : 312 M€. Fonds propres éligibles : 589 M€. '
     'Ratio de couverture SCR : 189%. MCR : 78 M€, couvert à 755%.',
     'E_GESTION_CAPITAL'),
    ('Provisions techniques Best Estimate : 2,1 Md€. Risk Margin (CoC 6%) : 45 M€. '
     'Valorisation conforme IFRS avec retraitements SII. Courbe des taux EIOPA.',
     'D_VALORISATION'),
]

doc_id = str(uuid.uuid4())
chunks_sfcr = [
    ChunkDocument(
        chunk_id=str(uuid.uuid4()), doc_id=doc_id, type_doc='SFCR',
        texte=t, page_debut=i+1, page_fin=i+1, section=s,
        position_dans_doc=i, nb_tokens=compter_tokens(t)
    )
    for i, (t, s) in enumerate(sfcr_contenu)
]

embs_sfcr = engine.embed_documents([c.texte for c in chunks_sfcr])
store.indexer_sfcr(chunks_sfcr, embs_sfcr.tolist(), 'Entreprise Demo SA', 2023)
print(f'SFCR indexé : {len(chunks_sfcr)} chunks dans ChromaDB')

In [ ]:
# Pipeline RAG — analyse Pilier 2 complet
llm = creer_llm_engine(mode='local')  # RuleBasedLLM offline
pipeline = RAGPipeline(engine, store, llm, top_k=3)

exigences_p2 = loader.get_exigences_par_pilier('P2')
print(f'Analyse Pilier 2 : {len(exigences_p2)} exigences...')

compteur = [0]
def progression(i, total, exig, res):
    compteur[0] = i

resultat = pipeline.analyser_sfcr(
    exigences=exigences_p2,
    doc_id=doc_id,
    entreprise='Entreprise Demo SA',
    annee=2023,
    callback_progression=progression,
)

print(f'✓ Analyse terminée en {resultat.temps_total_s:.2f}s')
print(f'  {resultat.nb_exigences_analysees} exigences | {resultat.nb_erreurs} erreurs')

In [ ]:
# Visualisation des résultats RAG
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Résultats RAG — Pilier 2 (Gouvernance & Contrôle)', 
             fontsize=13, fontweight='bold')

matchings = resultat.matchings
scores_finaux = [m.score_final for m in matchings]
scores_cos    = [m.score_similarite for m in matchings]
scores_llm    = [m.score_llm for m in matchings]
statuts       = [m.statut.value for m in matchings]
ids           = [m.id_exigence.split('-Art')[1] for m in matchings]

# Graphe 1 : Scores par exigence
x = range(len(matchings))
axes[0].bar(x, scores_finaux, color=[
    '#4CAF50' if s == 'COUVERT' else '#FF9800' if s == 'PARTIEL' else '#f44336'
    for s in statuts
], alpha=0.8)
axes[0].axhline(y=0.75, color='green', linestyle='--', alpha=0.7, label='Seuil COUVERT')
axes[0].axhline(y=0.45, color='orange', linestyle='--', alpha=0.7, label='Seuil PARTIEL')
axes[0].set_xlabel('Exigences')
axes[0].set_ylabel('Score final')
axes[0].set_title('Score de conformité par exigence')
axes[0].legend(fontsize=8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(ids, rotation=45, ha='right', fontsize=7)

# Graphe 2 : Cosinus vs LLM
axes[1].scatter(scores_cos, scores_llm, c=[
    '#4CAF50' if s == 'COUVERT' else '#FF9800' if s == 'PARTIEL' else '#f44336'
    for s in statuts
], s=100, alpha=0.8, edgecolors='black', linewidth=0.5)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Cosinus = LLM')
axes[1].set_xlabel('Score cosinus (vectoriel)')
axes[1].set_ylabel('Score LLM (sémantique)')
axes[1].set_title('Cosinus vs LLM\n(divergences = valeur ajoutée du LLM)')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
c = mpatches.Patch(color='#4CAF50', label='COUVERT')
p = mpatches.Patch(color='#FF9800', label='PARTIEL')
a = mpatches.Patch(color='#f44336', label='ABSENT')
axes[1].legend(handles=[c, p, a], fontsize=8)

# Graphe 3 : Statuts
nb_c = statuts.count('COUVERT')
nb_p = statuts.count('PARTIEL')
nb_a = statuts.count('ABSENT')
axes[2].pie(
    [nb_c, nb_p, nb_a],
    labels=[f'COUVERT\n({nb_c})', f'PARTIEL\n({nb_p})', f'ABSENT\n({nb_a})'],
    colors=['#4CAF50', '#FF9800', '#f44336'],
    autopct='%1.0f%%', startangle=90,
)
axes[2].set_title('Distribution des statuts')

plt.tight_layout()
plt.savefig('../docs/rag_resultats.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Affichage détaillé des résultats pour le prof
print('=== RAPPORT DE CONFORMITÉ DÉTAILLÉ — PILIER 2 ===')
print()

from src.ingestion.referentiel_solvabilite2 import THEMES

theme_courant = None
for res in sorted(resultat.resultats_exigences, key=lambda r: r.exigence.theme_id):
    if res.exigence.theme_id != theme_courant:
        theme_courant = res.exigence.theme_id
        theme = THEMES[theme_courant]
        print(f'\n{'─'*60}')
        print(f'[{theme_courant}] {theme.libelle} (poids pilier: {theme.poids_dans_pilier:.0%})')
        print(f'{'─'*60}')
    
    m = res.matching
    emoji = {'COUVERT':'✅','PARTIEL':'⚠️','ABSENT':'❌'}[m.statut.value]
    oblig  = res.exigence.niveau_obligation.value
    
    print(f'{emoji} [{oblig}] {m.id_exigence}')
    print(f'   Score: {m.score_final:.3f} = '
          f'0.4×{m.score_similarite:.3f}(cos) + 0.6×{m.score_llm:.3f}(llm)')
    if m.passage_trouve:
        print(f'   Passage: "{m.passage_trouve[:120]}..."')
    print(f'   Justification: {m.justification_llm[:120]}...')

In [ ]:
# Nettoyage
shutil.rmtree(demo_dir)
print('✓ Démonstration terminée — ressources nettoyées')
print()
print('PROCHAINE ÉTAPE : Phase 5 — Moteur de scoring')
print('  → Agrégation score exigence → thème → pilier → global')
print('  → Notation A/B/C/D avec pondérations métier')
print('  → Génération des recommandations')